In [0]:
%sql
USE e_commerce.bronze;

In [0]:
%run "../src/utils/configuration"

In [0]:
%run "../src/utils/read"

In [0]:
%run "../src/utils/rules"

In [0]:
%run "../src/utils/common_functions"

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, input_file_name

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2025-10-25")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

In [0]:
order_review_schema = StructType(fields=[
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", IntegerType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", TimestampType(), True),
    StructField("review_answer_timestamp", TimestampType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
path = f"{land_folder_path}/{v_file_date}/olist_order_reviews_dataset_{v_file_date}.csv"

In [0]:
options = {
    "header": True,
    "sep": ",",
    "quote": '"',
    "escape": '"',
    "multiLine": True,
    "mode": "PERMISSIVE",
    "columnNameOfCorruptRecord": "_corrupt_record"
}

In [0]:
df_order_review = read_data(path, format="csv", schema=order_review_schema, options=options)

In [0]:
reglas = []

In [0]:
if df_order_review.isEmpty():

    no_registros(df_order_review, "review_id", reglas, "order_review", v_file_date)

    df_reglas = spark.createDataFrame(reglas)
    df_reglas = df_reglas.withColumn("validation_date", current_timestamp())

    df_reglas.write.mode("append").format("delta").saveAsTable("bronze.log_transformation")

In [0]:
display(df_order_review.limit(2))

In [0]:
df_order_review = add_ingestion_timestamp(df_order_review)\
    .withColumn("environment", lit(v_environment))\
    .withColumn("file_date", lit(v_file_date))\
    .withColumn("source_name", input_file_name())

In [0]:
display(df_order_review.limit(2))

In [0]:
df_order_review_bronze = df_order_review.filter(col("_corrupt_record").isNull())
df_order_review_corrupted = df_order_review.filter(col("_corrupt_record").isNotNull())

In [0]:
display(df_order_review_bronze.limit(2))

In [0]:
try:
    df_order_review_bronze.write.format("delta").option("mergeSchema", "true").mode("append").partitionBy("file_date").option("path", f"{bronze_folder_path}/order_review").saveAsTable("order_review")

    df_order_review_corrupted.write.format("delta").option("mergeSchema", "true").mode("append").partitionBy("file_date").option("path", f"{bronze_folder_path}/order_review_corrupted").saveAsTable("order_review_corrupted")
except Exception as e:
    error_msg = handle_error(e, dbutils)
    print(error_msg)
    raise

In [0]:
%sql
SELECT file_date, COUNT(1) FROM bronze.order_review GROUP by file_date ORDER BY 1;

In [0]:
%sql
SELECT * FROM bronze.log_transformation;